# Advanced Prompting Architectures

Programmatically execute and validate advanced prompting architectures: Zero-Shot, Few-Shot, Chain-of-Thought (CoT), and the ReAct reasoning loop.

In [ ]:
import os
import re
from openai import OpenAI
from dotenv import load_dotenv

## Setup

In [ ]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def generate_response(prompt, stop=None):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        stop=stop
    )
    return response.choices[0].message.content.strip()

## Zero-Shot Prompting

In [ ]:
zero_shot_prompt = "Classify the sentiment of this text: 'The new optimization layer completely broke the pipeline.'"
zero_shot_output = generate_response(zero_shot_prompt)
print("Zero-Shot Output:\n", zero_shot_output)

## Few-Shot Prompting

In [ ]:
few_shot_prompt = """Classify the sentiment of the text.
Text: 'The battery life is terrible.'
Sentiment: Negative

Text: 'It works okay, nothing special.'
Sentiment: Neutral

Text: 'The new optimization layer completely broke the pipeline.'
Sentiment:"""
few_shot_output = generate_response(few_shot_prompt)
print("Few-Shot Output:\n", few_shot_output)

## Chain-of-Thought (CoT)

In [ ]:
cot_prompt = """Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A: Let's think step by step. Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11.

Q: The data center had 23 GPUs. If they allocated 20 to a training run and bought 6 more, how many available GPUs do they have?
A: Let's think step by step."""
cot_output = generate_response(cot_prompt)
print("CoT Output:\n", cot_output)

## ReAct Reasoning Loop

In [ ]:
react_system = """You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:
calculate:
e.g. Action: calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

Example session:
Question: What is 53 multiplied by 19?
Thought: I need to multiply 53 by 19.
Action: calculate: 53 * 19
PAUSE
Observation: 1007
Thought: I have the result.
Answer: 1007
"""

def execute_react(question, max_turns=5):
    prompt = f"{react_system}\n\nQuestion: {question}"
    
    for _ in range(max_turns):
        response = generate_response(prompt, stop=["Observation:"])
        print(response)
        prompt += f"\n{response}"
        
        action_match = re.search(r"Action: calculate: (.*)", response)
        if action_match:
            expression = action_match.group(1).strip()
            try:
                result = str(eval(expression))
            except Exception as e:
                result = str(e)
            
            observation = f"\nObservation: {result}"
            print(observation)
            prompt += observation
        elif "Answer:" in response:
            break

print("=== ReAct Execution ===")
execute_react("What is ((15 * 3) + 12) / 3?")